# Занятие 8. Python + sqlite3: функции для работы с базой данных

## Цель занятия

На прошлых занятиях мы изучили основные SQL-команды и аналитику.

Сегодня делаем следующий важный шаг:  
переносим SQL-запросы в **Python-функции**.

К концу занятия студент должен уметь:

- писать функции для подключения к базе данных;
- писать функции для создания таблицы;
- писать функции для добавления данных;
- писать функции для поиска данных;
- использовать параметры запросов `?`;
- понимать, зачем нужна защита от SQL injection;
- готовить код к переносу в VS Code и дипломный проект.

---

## Почему SQL нужно оформлять в функции

Плохой вариант:

```python
cursor.execute("SELECT * FROM users")
```

написан много раз в разных местах проекта.

Хороший вариант:

```python
def get_all_users():
    ...
```

Теперь код:
- проще читать;
- проще переиспользовать;
- проще переносить в проект;
- проще показывать на экзамене.

---

## Где это используется в реальной жизни

В реальных проектах SQL почти всегда спрятан внутри функций или методов.

Например:

- `add_user(...)`;
- `find_candidate_by_name(...)`;
- `get_orders_by_user(...)`;
- `get_training_report(...)`;
- `update_product_price(...)`.

Пользователь нажимает кнопку, а внутри вызывается функция, которая делает SQL-запрос.

---

## Что такое параметры запроса

Правильно:

```python
cursor.execute(
    "SELECT * FROM users WHERE city = ?",
    ("Казань",)
)
```

Неправильно:

```python
cursor.execute("SELECT * FROM users WHERE city = '" + city + "'")
```

Почему неправильно:
- можно сломать SQL-запрос;
- можно получить ошибку;
- можно открыть путь к SQL injection.

---

## Что такое SQL injection

SQL injection — это опасная ситуация, когда пользовательский ввод становится частью SQL-команды.

Например, пользователь вводит не имя, а кусок SQL-кода.

Поэтому в реальных проектах используют параметры `?`.

---

## Зачем это нужно в дипломе

На этом занятии студент начинает писать код как разработчик проекта.

В дипломе нужны функции:

- создать таблицы;
- добавить пользователя;
- найти запись;
- получить отчёт;
- обновить данные;
- удалить данные.

После этого занятия SQL-код можно переносить в файлы:

```text
src/db/connection.py
src/db/create_tables.py
src/db/queries.py
src/db/insert_data.py
```

---

## Связь с GitHub

На этом занятии студент должен сохранить:

- solved-ноутбук;
- TODO-ноутбук;
- Python-функции для sqlite3;
- будущие файлы `src/db/*.py`.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-08-python-sql-functions
```

Рекомендуемый commit:

```bash
git add .
git commit -m "feat: add python sqlite functions"
git push -u origin feature/lesson-08-python-sql-functions
```

## Ячейка 1. Подключаем sqlite3

Сегодня мы будем писать функции.  
Первая функция будет создавать подключение к базе данных.

In [ ]:
import sqlite3

print("sqlite3 подключен")

## Ячейка 2. Функция get_connection

Функция возвращает подключение к базе.

Такой подход удобен, потому что имя базы можно передавать параметром.

In [ ]:
def get_connection(db_name="python_sql_lesson08.db"):
    connection = sqlite3.connect(db_name)
    return connection

connection = get_connection()

print("Подключение создано")

assert connection is not None

## Ячейка 3. Функция create_users_table

Функция создаёт таблицу `users`.

Теперь создание таблицы можно вызывать одной строкой.

In [ ]:
def create_users_table(connection):
    cursor = connection.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        city TEXT,
        age INTEGER
    )
    """)
    connection.commit()

create_users_table(connection)

print("Таблица users создана")

## Ячейка 4. Функция clear_users

Эта функция очищает таблицу перед повторным запуском ноутбука.

Это нужно, чтобы данные не дублировались.

In [ ]:
def clear_users(connection):
    cursor = connection.cursor()
    cursor.execute("DELETE FROM users")
    connection.commit()

clear_users(connection)

print("Таблица users очищена")

## Ячейка 5. Функция add_user

Функция добавляет одного пользователя.

Обратите внимание: используем `?`, а не склеиваем SQL-строку вручную.

In [ ]:
def add_user(connection, name, city, age):
    cursor = connection.cursor()
    cursor.execute(
        "INSERT INTO users (name, city, age) VALUES (?, ?, ?)",
        (name, city, age)
    )
    connection.commit()

add_user(connection, "Анна", "Казань", 20)
add_user(connection, "Иван", "Москва", 21)
add_user(connection, "Ольга", "Казань", 19)

print("Пользователи добавлены")

## Ячейка 6. Функция get_all_users

Функция получает всех пользователей из таблицы.

In [ ]:
def get_all_users(connection):
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM users")
    return cursor.fetchall()

users = get_all_users(connection)

for user in users:
    print(user)

assert len(users) == 3

## Ячейка 7. Функция find_users_by_city

Функция ищет пользователей по городу.

Это уже похоже на реальную функцию для диплома.

In [ ]:
def find_users_by_city(connection, city):
    cursor = connection.cursor()
    cursor.execute(
        "SELECT * FROM users WHERE city = ?",
        (city,)
    )
    return cursor.fetchall()

kazan_users = find_users_by_city(connection, "Казань")

for user in kazan_users:
    print(user)

assert len(kazan_users) == 2

## Ячейка 8. Функция find_user_by_name

Функция ищет одного пользователя по имени.

Используем `fetchone()`, потому что ожидаем одну запись.

In [ ]:
def find_user_by_name(connection, name):
    cursor = connection.cursor()
    cursor.execute(
        "SELECT * FROM users WHERE name = ?",
        (name,)
    )
    return cursor.fetchone()

ivan = find_user_by_name(connection, "Иван")

print(ivan)

assert ivan[1] == "Иван"

## Ячейка 9. Функция update_user_age

Функция обновляет возраст пользователя.

Важно: всегда используем `WHERE`, чтобы не изменить все строки.

In [ ]:
def update_user_age(connection, name, new_age):
    cursor = connection.cursor()
    cursor.execute(
        "UPDATE users SET age = ? WHERE name = ?",
        (new_age, name)
    )
    connection.commit()

update_user_age(connection, "Иван", 22)

ivan_updated = find_user_by_name(connection, "Иван")

print(ivan_updated)

assert ivan_updated[3] == 22

## Ячейка 10. Функция delete_user и закрытие соединения

Функция удаляет пользователя по имени.

После проверки закрываем соединение.

In [ ]:
def delete_user(connection, name):
    cursor = connection.cursor()
    cursor.execute(
        "DELETE FROM users WHERE name = ?",
        (name,)
    )
    connection.commit()

delete_user(connection, "Ольга")

users_after_delete = get_all_users(connection)

for user in users_after_delete:
    print(user)

assert len(users_after_delete) == 2

connection.close()

print("Соединение закрыто")

# Итог занятия 8

Сегодня вы научились:

- писать Python-функции для SQL;
- создавать подключение к БД;
- создавать таблицу через функцию;
- добавлять данные через функцию;
- искать данные через функцию;
- обновлять и удалять записи;
- использовать безопасные параметры `?`.

## Что коммитить в GitHub

- solved-ноутбук;
- TODO-ноутбук;
- функции для работы с БД;
- будущие файлы `src/db/connection.py`, `queries.py`, `insert_data.py`.

```bash
git add .
git commit -m "feat: add python sqlite functions"
git push -u origin feature/lesson-08-python-sql-functions
```